## Kaggle Competition: Digit Recognizer for Dobong Campus

### Kaggle API Key 등록

- Kaggle Competition의 Code 탭에서 새 노트북을 열면, API Key를 연동하지 않아도 됩니다.
- 다음 셀은 Kaggle에서 새 노트북을 열 때, 기본적으로 제공되는 코드 블럭입니다.
- 다음 셀을 실행하면 이번 대회에서 제공하는 csv 파일의 경로를 콘솔에 출력합니다.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet18_Weights
from sklearn.model_selection import train_test_split
from trainer import Trainer, set_seed
from plt_rcs import *

In [ ]:
# 실행 환경 자동 감지 (Kaggle 내부 / 로컬 VSCode)
if os.path.exists('/kaggle/input'):
    BASE_INPUT = '/kaggle/input/competitions/digit-recognizer-for-dobong-campus'
    BASE_OUTPUT = '/kaggle/working'
else:
    BASE_INPUT = os.path.join(os.path.dirname(os.path.abspath('__file__')), '../data')
    BASE_OUTPUT = os.path.join(os.path.dirname(os.path.abspath('__file__')), '../data')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

set_seed(0)

print(f'환경: {"Kaggle" if os.path.exists("/kaggle/input") else "로컬"}')
print(f'device: {device}')


In [ ]:
# 훈련셋 csv 파일을 읽고 데이터프레임을 생성합니다.
train_df = pd.read_csv(filepath_or_buffer = os.path.join(BASE_INPUT, 'train.csv'))

In [ ]:
# # train_df의 처음 5행을 확인합니다.
# [참고] 첫 번째 열이 label이고, 나머지 784개 열은 pixel(image)입니다.
train_df.head()

In [ ]:
# train_df의 행 개수와 열 개수를 확인합니다.
train_df.shape

In [ ]:
# 시험셋 csv 파일을 읽고 데이터프레임을 생성합니다.
test_df = pd.read_csv(filepath_or_buffer = os.path.join(BASE_INPUT, 'test.csv'))

In [ ]:
# test_df의 처음 5행을 확인합니다.
# [참고] label 없이 784개 pixel(image) 열만 있습니다.
test_df.head()

In [ ]:
# test_df의 행 개수와 열 개수를 확인합니다.
test_df.shape

### 데이터셋 클래스 생성

- torch.utils.data.Dataset 클래스를 상속받아 일부 메서드를 추가합니다.

In [ ]:
import torchvision.transforms as transforms

mnist_mean = (0.1307,)
mnist_std  = (0.3081,)

class DigitDataset(Dataset):

    def __init__(self, dataframe, is_train=True):
        self.is_train = is_train

        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize(size=(128, 128)),
                transforms.RandomRotation(degrees=10),
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
                transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
                transforms.Normalize(mean=mnist_mean, std=mnist_std),
            ])
            self.labels = dataframe.iloc[:, 0].values
            self.images = dataframe.iloc[:, 1:].values
        else:
            self.transform = transforms.Compose([
                transforms.Resize(size=(128, 128)),
                transforms.Normalize(mean=mnist_mean, std=mnist_std),
            ])
            self.images = dataframe.values

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].reshape(28, 28)
        image = torch.tensor(data=image, dtype=torch.float32) / 255.0
        image = image.unsqueeze(dim=0)   # (1, 28, 28)
        image = self.transform(image)    # (1, 128, 128)

        if self.is_train:
            label = torch.tensor(data=self.labels[idx], dtype=torch.long)
            return image, label
        else:
            return image


### 훈련셋과 검증셋으로 분리

In [ ]:
# train_df를 8:2 비율로 train_sub와 valid_sub로 분리합니다.
train_sub, valid_sub = train_test_split(
    train_df, test_size = 0.2, stratify = train_df['label'], random_state = 0
)

In [ ]:
# train_sub, valid_sub, test_df로 파이토치 데이터셋을 생성합니다.
# [참고] 훈련셋과 검증셋은 is_train = True, 시험셋은 is_train = False입니다.
train_digit = DigitDataset(dataframe = train_sub, is_train = True)
valid_digit = DigitDataset(dataframe = valid_sub, is_train = True)
test_digit = DigitDataset(dataframe = test_df, is_train = False)

### 데이터 로더 생성

In [ ]:
bs = 128

In [ ]:
train_loader = DataLoader(dataset=train_digit, batch_size=bs, shuffle=True)
valid_loader = DataLoader(dataset=valid_digit, batch_size=bs, shuffle=False)
test_loader = DataLoader(dataset=test_digit, batch_size=bs, shuffle=False)

### 모델 정의 (ResNet18 전이학습)

- 사전학습된 ResNet18을 불러와 입력/출력 레이어를 교체합니다.
- `conv1`: 3채널 → 1채널 (흑백 이미지 대응)
- `fc`: 1000클래스 → 10클래스 (0~9 숫자 분류)

In [ ]:
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

In [ ]:
model

In [ ]:
# 입력 레이어 교체: 3채널 → 1채널 (흑백 이미지)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# 사전학습 레이어 전체 동결
for param in model.parameters():
    param.requires_grad = False

# layer3, layer4 동결 해제
for param in model.layer3.parameters():
    param.requires_grad = True
for param in model.layer4.parameters():
    param.requires_grad = True

# 출력 레이어 교체: Dropout + Linear (1000클래스 → 10클래스)
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(in_features=512, out_features=10),
)

model = model.to(device)

### 학습 루프

- 손실함수: `CrossEntropyLoss`, 옵티마이저: `Adam (lr=1e-3)`
- 매 epoch마다 훈련 손실과 검증 정확도를 출력합니다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    params=[
        {'params': model.layer3.parameters(), 'lr': 0.0001},
        {'params': model.layer4.parameters(), 'lr': 0.0001},
        {'params': model.fc.parameters(), 'lr': 0.0001},
    ]
)

trainer = Trainer(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=valid_loader,
    flatten=False
)

history = trainer.fit(n_epochs=10)

### 손실값 변화 그래프 조회

In [ ]:
trainer.plot_loss()

In [ ]:
trainer.plot_accuracy()

### 시험셋 예측

- 학습된 모델로 `test_loader`를 순회하며 예측값(`test_pred`)을 생성합니다.

In [ ]:
model.eval()
all_pred = []

with torch.no_grad():
    for images in test_loader:
        x = images.to(device)
        logits = model(x)
        y_pred = logits.argmax(dim=1)
        all_pred.append(y_pred.cpu())

test_pred = torch.cat(tensors=all_pred, dim=0)

In [ ]:
# Kaggle 제출용 데이터프레임을 생성합니다.
submission = pd.DataFrame(data = {
    'ImageId': range(1, len(test_pred) + 1),
    'Label': test_pred
})

In [ ]:
# submission을 csv 파일로 저장합니다.
# [주의] 제출 파일의 이름은 반드시 'submission.csv'여야 합니다.
# '/kaggle/working' 경로에 저장하면 오른쪽 메뉴를 통해 제출할 수 있습니다.
submission.to_csv(path_or_buf = os.path.join(BASE_OUTPUT, 'model_06.csv'), index = False)

### Kaggle Competition에 예측값 제출

- Colab에서는 Kaggle API를 이용하여 자동 제출이 가능합니다.
- Kaggle에서는 다음 과정을 거쳐 제출합니다.
  1. Kaggle 노트북에서 submission.csv 파일을 생성합니다.
  2. 오른쪽 메뉴바의 Output을 새로고침하여 submission.csv 파일을 확인합니다.
  3. 오른쪽 메뉴바의 Submit to competition에서 `Submit` 버튼을 클릭합니다.
  4. Version Name을 작성하고(선택), 하단의 `Submit` 버튼을 클릭하면 제출됩니다.

## End of Document